# Prodigy InfoTech - Machine Learning Internship
## Task-01: House Price Prediction using Linear Regression

### **Objective**
Implement a linear regression model to predict the prices of houses based on their **square footage**, **number of bedrooms**, and **bathrooms** using the Kaggle House Prices (Ames Housing) dataset.

---

### 1. Import Necessary Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Plot styling
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 11

### 2. Load and Explore the Dataset

In [ ]:
# Load dataset
df = pd.read_csv('data/train.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 3. Feature Selection & Descriptive Statistics
We select the target and core specified features:
- `GrLivArea`: Above grade (ground) living area square feet
- `BedroomAbvGr`: Bedrooms above grade
- `FullBath`: Full bathrooms above grade
- `HalfBath`: Half baths above grade
- `SalePrice`: Target property sale price in USD

In [ ]:
features = ['GrLivArea', 'BedroomAbvGr', 'FullBath', 'HalfBath']
target = 'SalePrice'

print("Summary Statistics:")
display(df[features + [target]].describe().round(2))

print("\nMissing values count:")
print(df[features + [target]].isnull().sum())

### 4. Exploratory Data Analysis & Visualizations

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(7, 5))
corr = df[features + [target]].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=1.0)
plt.title('Feature Correlation Matrix with SalePrice', fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Scatter Plot: Square Footage vs Price
plt.figure(figsize=(8, 5))
sns.regplot(x='GrLivArea', y='SalePrice', data=df, color='#1f77b4', line_kws={'color': 'red'})
plt.title('House Price vs Living Area Square Footage', fontsize=13, fontweight='bold')
plt.xlabel('Ground Living Area (Sq Ft)')
plt.ylabel('Sale Price ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### 5. Train-Test Split (80% Train, 20% Test)

In [ ]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size : {X_test.shape[0]} samples")

### 6. Model Training - Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model Intercept (Beta_0):", f"${model.intercept_:,.2f}")
for feat, coef in zip(features, model.coef_):
    print(f"Coefficient for {feat}: ${coef:,.2f} per unit")

### 7. Model Evaluation & Performance Metrics

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100

print("=" * 45)
print("         EVALUATION METRICS")
print("=" * 45)
print(f"Train R² Score : {r2_train:.4f} ({r2_train*100:.2f}%)")
print(f"Test R² Score  : {r2_test:.4f} ({r2_test*100:.2f}%)")
print(f"MAE            : ${mae:,.2f}")
print(f"MSE            : {mse:,.2f}")
print(f"RMSE           : ${rmse:,.2f}")
print(f"MAPE           : {mape:.2f}%")
print("=" * 45)

### 8. Actual vs Predicted Evaluation & Residual Analysis

In [ ]:
# Actual vs Predicted Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, color='#1f77b4', alpha=0.6, edgecolors='black', label='Predictions')
min_v, max_v = min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], 'r--', lw=2.5, label=f'Ideal Fit Line (R²={r2_test:.3f})')
plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.title('Actual vs Predicted House Prices', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Residuals Plot
residuals = y_test - y_test_pred
plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True, color='#2ca02c', bins=25)
plt.axvline(0, color='red', linestyle='--', lw=2)
plt.title('Residuals (Errors) Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Residual ($)')
plt.show()

### 9. Custom Sample Predictions
Let's test the model on custom house configurations:

In [ ]:
custom_houses = pd.DataFrame([
    {'GrLivArea': 1200, 'BedroomAbvGr': 2, 'FullBath': 1, 'HalfBath': 0},
    {'GrLivArea': 1800, 'BedroomAbvGr': 3, 'FullBath': 2, 'HalfBath': 1},
    {'GrLivArea': 2400, 'BedroomAbvGr': 4, 'FullBath': 2, 'HalfBath': 1},
    {'GrLivArea': 3200, 'BedroomAbvGr': 5, 'FullBath': 3, 'HalfBath': 2}
])

custom_preds = model.predict(custom_houses)
custom_houses['Predicted_SalePrice'] = [f"${p:,.2f}" for p in custom_preds]
custom_houses

### 10. Save the Trained Model

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/linear_regression_model.joblib')
print("Model successfully exported to models/linear_regression_model.joblib")